In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
from decimal import Decimal,ROUND_FLOOR

%matplotlib inline
%matplotlib notebook

## 1.Load file.

In [2]:
# load the csv file.
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Trial/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

## 2. Sewer System ###

In [4]:
iters = np.shape(date)[0] # total timestep.

In [5]:
ss_measure = 0 # we do not consider 'measure' for the time being.

### 2.1 Build up using default settings in excel.

#### 2.2.1 Input data preparation

__a.__ r_swds_pr, r_swds_cp, r_swds_op

In [6]:
r_swds_pr = [0,0,0,0,0,0,0.216,0.252337663,0.356064935,0,0,0,0.02725974,0.102064935,0.102064935,0,0,3.98986E-17,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0.420850651,0.110344156,0,0,0.107012987,0,0,0,0.127,0,0.127,0.762,2.54,3.81,6.35,2.286,0.118805511,4.16334E-17,0
,0,0]
r_swds_cp = [0,0,0,0,0,0,0.432,0.504675325,0.71212987,0,0,0,0.05451948,0.20412987,0.20412987,0,0,7.97973E-17,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.841701301,0.220688312,0,0,0.214025974,0,0,0,0.254,0,0.254,1.524,5.08,7.62,12.7,4.572
,0.237611021,8.32667E-17,0,0,0]
r_swds_op = [0,0,0,0,0,0,0.390333333,0.463008658,0.670463203,0,0,0,0.012852813,0.162463203,0.162463203,0,0,7.97973E-17,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.800034634,0.179021645,0,0,0.172359307,0,0,0,0.212333333,0,0.212333333,1.482333333,5.038333333,7.578333333,12.65833333
,4.530333333,0.195944354,8.32667E-17,0,0,0]
r_mss_pr, r_mss_cp, r_mss_op = np.zeros(iters), np.zeros(iters), np.zeros(iters)

__b.__ open water level, taken from excel.

#### 2.2.2 Using Arrays to build up the structure first.

In [12]:
# areas for different area
pr_area = 1560
cp_area = 803.3906406
op_area = 481.6093594
pr_discfrac = 0.5
cp_discfrac, op_discfrac = 0, 0
tot_disc_area = pr_area * pr_discfrac + cp_area * cp_discfrac + op_area * op_discfrac
sdws_area = (pr_area + cp_area + op_area - tot_disc_area) * 1 # 1 is the storm drainage fraction 100%
mss_area = (pr_area + cp_area + op_area - tot_disc_area) * 0 # 0 is the mss drainage fraction 0 %

In [14]:
delta_t = 1 / 24

r_sdws = np.zeros(iters)
r_meas_sdws = np.zeros(iters)
r_mss = np.zeros(iters)
r_meas_mss = np.zeros(iters)
q_ow_sdws = np.zeros(iters)
stor_sdws = np.zeros(iters)
so_sdws = np.zeros(iters)

q_out_mss = np.zeros(iters)
stor_mss = np.zeros(iters)
q_ow_mss = np.zeros(iters)
so_mss = np.zeros(iters)

q_ow_mss = np.zeros(iters)
so_sdws = np.zeros(iters)
stor_sdws = np.zeros(iters)
stor_mss = np.zeros(iters)

q_ow_sdws_cap = 55.1
q_ow_mss_cap = 48.1
q_out_mss_cap = 26.3
stor_sdws_cap = 2
stor_mss_cap = 9
ow_area = 300

t = 1 


while t <= iters - 1:
    
    #  Total runoff to storm water drainage system [mm].
    if sdws_area == 0:
        r_sdws[t] = 0
    else:
        r_sdws[t] = (r_swds_pr[t] * pr_area + r_swds_cp[t] * cp_area + r_swds_op[t] * op_area) / swds_area
    
    # Inflow from measure area (if applicable) [mm]
    r_meas_sdws[t] = 0
    
    # Total runoff to mixed sewer system [mm].
    if mss_area != 0:
        r_mss[t] =  (r_mss_pr[t] * pr_area + r_mss_cp[t] * cp_area + r_mss_op[t] * op_area) / mss_area
    else:
        r_mss[t] = 0
        
    # Inflow from measure area (if applicable) [mm]     
    r_meas_mss[t] = 0
    
    # Outflow from storm water drainage system to open water [mm]
    if swds_area == 0:
        q_ow_sdws[t] = 0
    else:
        if ow_area == 0:
            q_ow_sdws[t] = min(stor_sdws[t-1] + r_sdws[t] + r_meas_sdws[t] + so_sdws[t-1], q_ow_sdws_cap)
        else:
            q_ow_sdws[t] = min(stor_sdws[t-1] + r_sdws[t] + r_meas_sdws[t] + 0, q_ow_sdws_cap)
            
    # Discharge from mixed sewer system to Waste Water Treatment Plant (WWTP) during the current time step [mm]
    if mss_area == 0:
        q_out_mss[t] = 0
    else:
        if ow_area == 0:
            q_out_mss[t] = min(stor_mss[t-1] + r_mss[t] + r_meas_mss[t] + so_mss[t-1], q_out_mss_cap)
        else:
            q_out_mss[t] = min(stor_mss[t-1] + r_mss[t] + r_meas_mss[t] + 0, q_out_mss_cap)
            
            
    
    # Outflow from mixed sewer system to open water [mm]
    if mss_area == 0:
        q_ow_mss[t] = 0
    else:
        if ow_area == 0:
            q_ow_mss[t] = max(0, min(stor_mss[t-1] + r_mss[t] + r_meas_mss[t] - q_out_mss[t] + so_mss[t-1], q_ow_mss_cap - q_out_mss_cap))
        else:
            q_ow_mss[t] = max(0, min(stor_mss[t-1] + r_mss[t] + r_meas_mss[t] - q_out_mss[t] + 0, q_ow_mss_cap - q_out_mss_cap))
            
    
    # Overflow of storm water drainage system [mm]
    if swds_area == 0:
        so_sdws[t] = 0
    else:
        if ow_area == 0:
            so_sdws[t] = max(0, stor_sdws[t-1] + r_sdws[t] + r_meas_sdws[t] - q_ow_sdws[t] - stor_sdws_cap + so_sdws[t-1])
        else:
            so_sdws[t] = max(0, stor_sdws[t-1] + r_sdws[t] + r_meas_sdws[t] - q_ow_sdws[t] - stor_sdws_cap + 0)
            
            
    # Overflow of mixed sewer system [mm]
    if mss_area == 0:
        so_mss[t] = 0
    else:
        if ow_area == 0:
            so_mss[t] = max(0, stor_mss[t-1] + r_mss[t] + r_meas_mss[t] - q_out_mss[t] - q_ow_mss[t] - stor_mss_cap + so_mss[t-1])
        else:
            so_mss[t] = max(0, stor_mss[t-1] + r_mss[t] + r_meas_mss[t] - q_out_mss[t] - q_ow_mss[t] - stor_mss_cap + 0)
    
    # Storage in the storm water drainage system at the end of the current time step [mm]
    if sdws_area == 0:
        stor_sdws[t] = 0
    else:
        if ow_area == 0:
            stor_sdws[t] = max(0, stor_sdws[t-1] + r_sdws[t] + r_meas_sdws[t] - q_ow_sdws[t] - (so_sdws[t] - so_sdws[t-1]))
        else:
            stor_sdws[t] = max(0, stor_sdws[t-1] + r_sdws[t] + r_meas_sdws[t] - q_ow_sdws[t] - so_sdws[t])
    
    # Storage in the mixed sewer system at the end of the current time step [mm]
    if mss_area == 0:
        stor_mss[t] = 0
    else:
        if ow_area == 0:
            stor_mss = max(0, stor_mss[t-1] + r_mss[t] + r_meas_mss[t] - q_out_mss[t] - q_ow_mss[t] - (so_mss[t] - so_mss[t-1]))                             
        else:
            stor_mss = max(0, stor_mss[t-1] + r_mss[t] + r_meas_mss[t] - q_out_mss[t] - q_ow_mss[t] - so_mss[t])
    
    t += 1
    
    

#print(r_sdws)
filename = 'Results_sewersystem_arraybuildup_test.csv'
np.savetxt('sol/' + filename, np.c_[r_sdws, r_meas_sdws, r_mss, r_meas_mss, q_ow_sdws, q_out_mss, q_ow_mss, so_sdws, so_mss, stor_sdws, stor_mss], fmt = "%.8f", delimiter=',', header = 'r_sdws, r_meas_sdws, r_mss, r_meas_mss, q_ow_sdws, q_out_mss, q_ow_mss, so_sdws, so_mss, stor_sdws, stor_mss') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

#print('The results have been validated with excel.')

#### 2.2.3 Using Class to build up the module

In [13]:
class Groundwater:
    def __init__(self, gw_area, seep_def = 0, w = 100, vc = 20000, h_deepgw = 21.5, flux = 1, init_gwl = 1.5, croptype = 2, soiltype = 1):
        
        # state
        self.init_gwl = init_gwl
        
        # parameter
        self.gw_area =  gw_area
        self.seep_def = seep_def
        self.w =  w
        self.vc = vc
        self.h_deepgw = h_deepgw
        self.flux = flux
        self.soiltype = soiltype
        self.croptype = croptype
        self.prev_gwl = init_gwl
        self.prev_gwl_sl = 0
    
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are current precipitation and evaporation.'
    
    def sol(self, p_uz_gw, uz_area, p_op_gw, prev_owl, op_area, delta_t = 1 / 24): 
        
        # sum_p_gw
        sum_p_gw = (p_uz_gw * uz_area + p_op_gw * op_area) / self.gw_area

        # Inflow from measure area (if applicable), set as 0 for the time being
        r_meas_gw = 0
        
        # gwl_up
        c = float(self.prev_gwl)
        if c>= 0.0 and c <= 2.5:
            c = float(Decimal(str(c)).quantize(Decimal('.1'), rounding=ROUND_FLOOR))
        elif c < 3.0:
            c = 2.5
        elif c < 5.0:
            c = int(c)
        elif c <= 10:
            c = 5.0
        else:
            c = 10.0
        gwl_up = c
    
        # gwl_low
        if gwl_up < 2.5:
            gwl_low = round(gwl_up + 0.1, 2)
        elif gwl_up < 3:
            gwl_low = 3
        elif gwl_up < 4:
            gwl_low = 4
        elif gwl_up < 5:
            gwl_low = 5
        else:
            gwl_low = 10
        
        # Storage coefficient of the groundwater for the current time step
        if self.prev_gwl < 10:
            sc_gw = SoilSelector(self.soiltype, self.croptype, gwl_low)['stor_coef'].values + (gwl_low - self.prev_gwl) / (gwl_low - gwl_up) * (SoilSelector(self.soiltype, self.croptype, gwl_up)['stor_coef'].values - SoilSelector(self.soiltype, self.croptype, gwl_low)['stor_coef'].values)
        else:
            sc_gw = SoilSelector(self.soiltype, self.croptype, 10)['stor_coef'].values
        
        # Groundwater level at the end of the current time step [m-SL].
        if self.seep_def > 0.5:
            h_gw = -(((sum_p_gw + r_meas_gw) / 1000 * self.w * self.vc - self.h_deepgw * self.w - prev_owl * self.vc) / (self.w + self.vc) + (-(self.prev_gwl + self.prev_gwl_sl) - ((sum_p_gw + r_meas_gw) / 1000 * self.w * self.vc - self.h_deepgw * self.w - prev_owl * self.vc) / (self.w + self.vc)) * np.exp(- delta_t * (self.w + self.vc) /(sc_gw * self.w * self.vc)))
        else:
            h_gw = - (self.w * (((sum_p_gw + r_meas_gw) - self.flux) / 1000) - prev_owl + (-(self.prev_gwl + self.prev_gwl_sl) - (self.w * (((sum_p_gw + r_meas_gw)- self.flux) / 1000) - prev_owl)) * np.exp(- delta_t / (sc_gw * self.w )))         
       
        # Downward seepage flux to deep groundwater during current time step.
        if self.seep_def < 0.5:
            s_out = delta_t * self.flux
        else:
            s_out = 1000 * (self.h_deepgw - 0.5 * (h_gw + (self.prev_gwl + self.prev_gwl_sl))) / self.vc * delta_t
        
        # Groundwater drainage to the open water for the current time step [mm].
        d_ow = sum_p_gw + r_meas_gw - s_out - sc_gw * (self.prev_gwl + self.prev_gwl_sl -  h_gw) * 1000        
 
        # Groundwater level below surface level at the end of the current time step [m-SL].
        gwl = max(0, self.prev_gwl - (sum_p_gw + r_meas_gw - s_out - d_ow) / (1000 * sc_gw))
    
        # Groundwater level above surface level at the end of the current time step [m-SL]
        gwl_sl = -1 * max(0, (0 - (self.prev_gwl - (sum_p_gw + r_meas_gw - s_out - d_ow)/(1000 * sc_gw))) * sc_gw)          
        
            
        # update state
        self.prev_gwl = gwl
        self.prev_gwl_sl = gwl_sl
         
        return sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl

In [14]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(2, 1, 1.5)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.5]
gwl_sl= [0]

# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 0, w = 100, vc = 20000, h_deepgw = 21.5, flux = 1, init_gwl = 1.5, croptype = 2, soiltype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_sewersystem_c1s1.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.


### 2.3 Validate with excel with two sets of coefficients.

#### 2.3.1 C2S1

Drainage resistance 80; 
seepage = 0;
flux = 2;
init_gwl = 1.6;
h_deepgw = 23;
flow resistence  = 30000;

__Note that__ when you change the __draiange resistance, flux and init_gwl__, the gwl is automatically changed, __so the input percolation from unsaturated zone is also changed.

In [16]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(2, 1, 1.6)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.6]
gwl_sl= [0]


# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 0, w = 80, vc = 30000, h_deepgw = 23, flux = 2, init_gwl = 1.6, croptype = 2, soiltype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_Groundwater_c2s1.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.


#### 2.3.2 C2S2

Drainage resistance 120; 
seepage = 1;
flux = 1.5;
init_gwl = 1.3;
h_deepgw = 20;
flow resistence  = 25000;
Soiltype = 3 
croptype = 1

In [17]:
p_uz_gw =[0,0,0.012454488,0.01235691,0.012314248,0.520271547,1.577586236,0.512277858,0.787725029,-0.036021096,-0.032474091,-0.032532207
,0.221298711,0.236757085,0.236652687,-0.03386665,-0.032724943,0.0030632,0.011826368,0.011748062,0.01170814,0.011668236,0.011628513
,0.011588971,0.011549608,0.011510423,0.011471415,0.011432582,0.011393924,0.011355439,0.009322423,-0.018633047,-0.01855179,-0.018590603
,-0.01862873,-0.018666688,-0.018704478,-0.0187421,-0.018779555,1.32458118,0.249958166,0.003909225,0.010918914,0.289205683,0.009622427
,0.010789775,0.010748858,0.293614775,0.009466674,0.293553116,1.706810486,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875,1.8875
,1.805383626]
p_op_gw = [0,0,0,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0.041666667,0.041666667,0.041666667,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
,0,0,0,0,0,0,0,0,0,0.041666667,0.041666667,0,0,0.041666667,0,0,0,0.041666667,0,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667,0.041666667
,0.041666667,0,0,0,0]

In [18]:
t = 1

sum_p_gw = [0] 
r_meas_gw = [0]
gwl_up = [0]
gwl_low = [0]
sc_gw = [SoilSelector(3, 1, 1.3)['stor_coef'].values]
h_gw = [0]
s_out = [0]
d_ow = [0]
gwl = [1.3]
gwl_sl= [0]


# Specify the parameter or use the default setting.
m = Groundwater(gw_area, seep_def = 1, w = 120, vc = 25000, h_deepgw = 20, flux = 1.5, init_gwl = 1.3, soiltype = 3, croptype = 1)

while t <= iters-1:
    # only loop sol(), not repeat creating new object.ow_area = 300
    sol = m.sol(p_uz_gw[t], 6855, p_op_gw[t], prev_owl = owl[t-1], op_area = 481.6093594, delta_t = 1 / 24)
    
    sum_p_gw.append(sol[0])
    r_meas_gw.append(sol[1])
    gwl_up.append(sol[2]) 
    gwl_low.append(sol[3]) 
    sc_gw.append(sol[4])
    h_gw.append(sol[5])
    s_out.append(sol[6])
    d_ow.append(sol[7])
    gwl.append(sol[8])
    gwl_sl.append(sol[9])

    # print('time step', t)
    t += 1
    
filename = 'Results_Groundwater_c2s2.csv'
np.savetxt('sol/' + filename, np.c_[sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl], fmt = "%.8f", delimiter=',', header = 'sum_p_gw, r_meas_gw, gwl_up, gwl_low, sc_gw, h_gw, s_out, d_ow, gwl, gwl_sl') 

# Insert the Date column for locating purposes.
df = pd.read_csv('sol/' + filename)
df.insert(0, 'Date', date)
df.to_csv('sol/' + filename)

print('The results have been validated.')

The results have been validated.
